# WP set encoder — Kaggle GPU training

Free path for training a win-probability model (PLAN.md Phase 4+). 30 GPU-hours/week, no balance
to manage. A run of this size takes **2–3 minutes**, versus ~45 on the laptop.

**Before running:**
1. Locally: `scripts/cloud/pack_training.sh reg_mc wp-v1` and unpack it, or just take the directory
   `data/features/reg_mc/wp-v1` (~110 MB).
2. Add it as a **private Kaggle Dataset** (Datasets → New Dataset → upload the folder). Name it
   `vgc-wp-features` so the path below matches, or edit `DATA` in the next cell.
3. Notebook settings: **Accelerator = GPU T4 x2** (or P100), **Internet off** is fine — nothing is
   downloaded.
4. Add this notebook's `set_torch.py`: upload `src/vgc/wp/set_torch.py` as a second Dataset (or paste
   it into a cell). `SRC` below points at it.

**After running:** download `model.onnx` + `train.json` from the output, put them in
`models/wp/reg_mc/<version>/`, then locally run `vgc wp card`, `vgc wp calibrate` and `vgc wp eval`.

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

INPUT = Path("/kaggle/input")
OUT   = Path("/kaggle/working/model")

def find(filename, hint):
    """Locate a file under /kaggle/input rather than hardcoding a path.

    Kaggle does not preserve the directory layout you upload: a folder pushed with
    `--dir-mode zip` is extracted with its contents at the dataset root, so a path like
    `<dataset>/wp-v1/info.json` silently becomes `<dataset>/info.json`. Searching costs
    nothing and survives that, and a clear error here beats one six cells later."""
    hits = sorted(INPUT.glob(f"**/{filename}"))
    if not hits:
        raise FileNotFoundError(
            f"no {filename} under {INPUT} -- did you attach the {hint} dataset?\n"
            + "\n".join(f"  {p}" for p in sorted(INPUT.glob('*/*'))[:20]))
    return hits[0]

DATA = find("info.json", "features").parent
SRC  = find("set_torch.py", "trainer")
print("features:", DATA)
print("trainer: ", SRC)

pkg = Path("/kaggle/working/src/vgc/wp")
pkg.mkdir(parents=True, exist_ok=True)
for p in (pkg.parent.parent, pkg.parent, pkg):
    (p / "__init__.py").touch()
shutil.copy(SRC, pkg / "set_torch.py")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
info = json.loads((DATA / "info.json").read_text())
print("rows:", info["rows"])
for f in ("train.npz", "val.npz", "vocab.json"):
    assert (DATA / f).exists(), f"missing {f} in {DATA}"

## One run

Flags mirror `scripts/cloud/run_training.sh`. On a GPU, use a larger batch (`--bs 2048`): the model
is small and small batches leave the GPU idle.

In [ ]:
def train(name, *flags, epochs=12, bs=2048, human_weight=4):
    """One run. `--human-weight > 1` also switches checkpoint selection to the human validation
    rows (set_torch picks the epoch by `val_wp_logloss_human`), which is the point: validation is
    72% self-play, and the epoch that minimises overall val loss is measurably worse on human
    play. Ranking a sweep on the overall number picks the wrong model."""
    out = OUT / name
    cmd = [sys.executable, "-u", "-m", "vgc.wp.set_torch", "--data", str(DATA), "--out", str(out),
           "--device", "auto", "--epochs", str(epochs), "--bs", str(bs),
           "--human-weight", str(human_weight), *map(str, flags)]
    t0 = time.time()
    subprocess.run(cmd, cwd="/kaggle/working", env={**os.environ, "PYTHONPATH": "/kaggle/working/src"}, check=True)
    r = json.loads((out / "train.json").read_text())
    print(f"{name}: human {r.get('val_wp_logloss_human')} | all {r['val_wp_logloss']} | "
          f"selected on {r.get('selected_on')}, best epoch {r['best_epoch']}, {time.time()-t0:.0f}s")
    return r


train("wp-v1-set", "--d", 128, "--layers", 3, "--dropout", 0.2, "--id-dropout", 0.5, "--id-dropout-preview", 0.0)

## A sweep

The real reason to use a GPU: try several configurations and keep the best.

**What this sweep is testing.** Snapshots inside one battle share an outcome, and self-play plays
each pairing 20 times, so the cheapest way to cut training loss is to memorise which team beat
which. `--id-dropout` blocks that by hiding a Pokémon's identity at random. But team identity is
the *only* input that exists at team preview, so a single uniform rate cannot serve both ends —
measured on Reg M-C: at 0.5 the spectator ECE and bring gates pass and preview WP collapses
(spread sd 0.146 against a simulated 0.419, correlation -0.021 on held-out teams); at 0.2 that
reverses. `--id-dropout-preview` splits the knob so preview/bring rows keep their identities
while turn rows stay masked. The `uniform-*` configs are the controls that reproduce both ends.

Ranked by **human-row validation loss**, not overall: see the note in the previous cell.

In [ ]:
configs = {
    # hypothesis: keep identity where it is the whole input, mask it where the board exists
    "split-0.5/0.0":  ("--d", 128, "--layers", 3, "--dropout", 0.2, "--id-dropout", 0.5, "--id-dropout-preview", 0.0),
    "split-0.7/0.0":  ("--d", 128, "--layers", 3, "--dropout", 0.2, "--id-dropout", 0.7, "--id-dropout-preview", 0.0),
    "split-0.5/0.2":  ("--d", 128, "--layers", 3, "--dropout", 0.2, "--id-dropout", 0.5, "--id-dropout-preview", 0.2),
    "split-small":    ("--d", 64,  "--layers", 2, "--dropout", 0.3, "--id-dropout", 0.5, "--id-dropout-preview", 0.0),
    "split-wide":     ("--d", 192, "--layers", 3, "--dropout", 0.3, "--weight-decay", 0.2,
                       "--id-dropout", 0.6, "--id-dropout-preview", 0.0),
    # controls: the two ends of the uniform knob, already measured locally
    "uniform-0.5":    ("--d", 64,  "--layers", 2, "--dropout", 0.3, "--id-dropout", 0.5),
    "uniform-0.2":    ("--d", 64,  "--layers", 2, "--dropout", 0.3, "--id-dropout", 0.2),
}
results = {name: train(name, *flags) for name, flags in configs.items()}

key = lambda r: r.get("val_wp_logloss_human") or r["val_wp_logloss"]
print(f"\n{'config':16} {'human':>8} {'all':>8}  epoch")
for name, r in sorted(results.items(), key=lambda kv: key(kv[1])):
    print(f"{name:16} {key(r):8.5f} {r['val_wp_logloss']:8.5f}  {r['best_epoch']}")

print("\nValidation loss does not decide this: the gates are measured locally on held-out human\n"
      "games, and preview is not in this number at all. Bring the top few home and run\n"
      "`vgc wp eval` + `vgc wp check-preview` on each.")

## Collect

Download the zip, unpack into `models/wp/reg_mc/`, then locally, **for each candidate**:

```bash
vgc wp card      --version <name>
vgc wp calibrate --version <name>
vgc wp eval      --version <name> --baseline wp-v1-gbt --baseline wp-v1-logistic --baseline constant
vgc wp check-preview --version <name>      # the gate this sweep exists to fix
vgc wp registry                            # prints gate pass/fail per model
```

Evaluation always runs locally against the frozen held-out split — nothing here decides that.
`vgc wp eval` writes the gate verdicts into the model card, so a model that misses one says so.

In [ ]:
shutil.make_archive("/kaggle/working/wp-models", "zip", OUT)
print(sorted(p.name for p in OUT.glob("*")))